In [21]:
import numpy as np
import pandas as pd
import re 
from datetime import datetime
from pathlib import Path
from scipy.stats import linregress
from matplotlib import pyplot as plt
class Frontage:
    def __init__(self, input_file: str, assay_date: datetime):
        self.input_file = input_file
        self.filename = Path(input_file).parts[-1]
        assert self.filename.endswith(".xlsx"), "Incoming data must be an Excel file"
        self.assay_date = assay_date
    
    def __repr__(self):
        return f"Frontage assay handler built on file: '{self.filename}' on day: {self.assay_date}"

class FrontageStability(Frontage):
    def __init__(self, input_file, assay_date):
        super().__init__(input_file, assay_date)
        self.scraped_data = None 

        self.excel_file = pd.ExcelFile(input_file)
        self.sheet_names = self.excel_file.sheet_names
        
        self.target_sheet = self.check_sheet_names()

        self.data = self.read_data()
        self.check_columns()
        self.handle_outliers()
        self.scraped_data_transformation()
        self.calculate_remaining()
        self.calculate_regression()
        self.rationalize_half_life()
    
    def __repr__(self):
        return super().__repr__()
    
    def read_data(self):
        """Read in data and store."""
        df = pd.read_excel(self.input_file, sheet_name=self.target_sheet, usecols=range(22))
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        df = df.loc[~(df.isna().all(axis=1))]
        return df

    def check_sheet_names(self):
        """Make sure there is a sheet name titled 'Raw' within the file."""

        # looking for any sheet resembling "raw data"
        for sheet in self.sheet_names:
            if re.search(r"[Rr]+aw", sheet):
                print(f"Valid sheet name '{sheet}' identified!")
                return sheet
        raise ValueError("Looking for sheet named 'Raw' but none found")
    
    def check_columns(self):
        """Change column headers and check to see if mandatory columns are present."""
        df = self.data.copy()
        
        # look for column names
        #TODO: these columns are those used in the first data delivery - do they change?
        mandatory_columns = ["cmpd_id", "sample_id", "is_area", "peak_area", "matrix", "notes"]
        for m in mandatory_columns:
            if m not in df.columns:
                raise ValueError(f"Column '{m}' not found in data sheet.")
            
        print("All needed column names found!")
        self.scraped_data = df[mandatory_columns]
            
        return 
    
    def handle_outliers(self):
        """Dictate whether data is outlier based on CRO recommendation."""
        df = self.scraped_data.copy()
        
        # fill empty string
        df.loc[:, "notes"] = df.notes.fillna("")

        # set new column with values
        df.loc[(df.notes.str.contains(r"[Oo]mitted", regex=True)), "outlier"] = True 
        df.loc[:, "outlier"] = df.outlier.fillna(False)

        # drop notes column, not needed
        df = df.drop("notes", axis=1)
        self.scraped_data = df
    
    def scraped_data_transformation(self):
        """Rename scraped data columns, transform data to prepare for calculations."""
        df = self.scraped_data.copy()

        # rename columns to resemble those used in CDD
        df = df.rename(columns=dict(
            zip(
                ["cmpd_id", "sample_id", "is_area", "peak_area"],
                ["molecule_name", "time", "is_area", "cmpd_area"]
            )
        ))
        
        # log10 transform values
        for column in ["is_area", "cmpd_area"]:
            component, suffix = column.split("_")
            df.loc[:, f"{component}_log10_{suffix}"] = np.log10(df[column])
        
        self.scraped_data = df 
        return 
    
    def calculate_remaining(self):
        """Calculate percent remaining and apply natural log to data."""
        df = self.scraped_data.copy()

        

        df = df.set_index(["molecule_name", "matrix", "time"]).sort_index()
        df = df.reset_index("time")

        # calculate is/cmpd ratio
        df.loc[:, "area_ratio"] = df.cmpd_area / df.is_area

        # for each molecule, use ratio to determine percent remaining
        for mol in df.index.unique():

            data = df.loc[mol, :]
            
            ref_val = data[data.time==0].area_ratio
            percent_remaining = data.area_ratio / ref_val
            
            # transform values using the natural log
            ln_percent_remaining = np.log(percent_remaining)


            # insert into dataframe
            df.loc[mol, "percent_remaining"] = percent_remaining
            df.loc[mol, "ln_percent_remaining"] = ln_percent_remaining
            
        self.scraped_data = df.reset_index()
        
        return 
    
    def calculate_regression(self):
        """Grab time and natural log data, filter, and perform linear regression."""

        df = self.scraped_data.copy()
        df = df.set_index(["molecule_name", "matrix"]).sort_index()

        for mol in df.index.unique():

            data = df.loc[mol, :]
            data = data[data.outlier==False]
            x, y = data.time.to_numpy(), data.ln_percent_remaining.to_numpy()


            m, b, r, _, _ = linregress(x, y)
            m, b, r = np.round(np.array([m, b, r]), 4)
            
            df.loc[mol, "rate_constant"] = m
            df.loc[mol, "linear_fit"] = -r
            df.loc[mol, "half_life"] = np.round(-0.693/m, 1)

        self.scraped_data = df.reset_index()
        
    def rationalize_half_life(self):
        """If half-life is longer than assay time, correct the values."""

        df = self.scraped_data.copy()
        max_time = df.time.max()

        df.loc[(df.half_life>max_time) | (df.half_life<0), "rational_half_life"] = max_time
        df.loc[:, "rational_half_life"] = df.rational_half_life.fillna(df.half_life)

        self.scraped_data = df

In [22]:
file = '/Users/delafield/Library/CloudStorage/GoogleDrive-delafield@calicolabs.com/My Drive/Projects/ADME_CRO/2025-11-10 Calico Liver Microsome Stability Result.xlsx'
file = '/Users/delafield/Library/CloudStorage/GoogleDrive-delafield@calicolabs.com/My Drive/Projects/ADME_CRO/2025-11-11 Calico Plasma Stability Result v2.xlsx'
file = '/Users/delafield/Library/CloudStorage/GoogleDrive-delafield@calicolabs.com/My Drive/Projects/ADME_CRO/2025-11-11 Calico SIF-SGF Stability Result v2.xlsx'
assay_date = datetime(2025, 11, 11)
fs = FrontageStability(file, assay_date)
fs.scraped_data.reset_index()[["molecule_name", "matrix", "rate_constant", "rational_half_life"]].groupby(["matrix", "molecule_name"]).mean()

Valid sheet name 'Raw Data' identified!
All needed column names found!


/var/folders/wc/8zzbrm9s0g3dm6svj8ztxlhr0000gp/T/ipykernel_54351/1479762598.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[:, "outlier"] = df.outlier.fillna(False)


rate_constant  rational_half_life
matrix              molecule_name                                   
FasSGF + Pepsin     CP-5a-1174           -0.0002               240.0
                    CP-5a-1193           -0.0001               240.0
                    CP-5a-1262           -0.0011               240.0
                    CP-5a-470             0.0002               240.0
                    CP-5a-838             0.0002               240.0
                    CP-5a-841             0.0003               240.0
                    CP-5a-850            -0.0003               240.0
                    Somatostatin         -0.0113                61.3
FasSIF + Pancreatin BAN                  -0.1210                 5.7
                    CP-5a-1174           -0.1665                 4.2
                    CP-5a-1193           -0.1671                 4.1
                    CP-5a-1262           -0.1106                 6.3
                    CP-5a-470            -0.2541                 2.7
                    CP-5a-838            -0.0050               138.6
                    CP-5a-841            -0.0359                19.3
                    CP-5a-850            -0.0005               240.0

In [28]:
fs.scraped_data.molecule_name.unique().tolist()
mol_map = {'CP-5a-1174': 'CAL-0011526',
'CP-5a-1193': 'CAL-0011553',
'CP-5a-1262': 'CAL-0011599',
'CP-5a-470': 'CAL-0010270',
'CP-5a-838': 'CAL-0010995',
'CP-5a-841': 'CAL-0010971',
'CP-5a-850': 'CAL-0010967',
'CP-8-176': 'CAL-0012008',
'CP-8-178': 'CAL-0012010',
'CP-5a-1174': 'CAL-0011526', 
'CP-5a-1193': 'CAL-0011553', 
'CP-5a-1262': 'CAL-0011599', 
'CP-5a-470': 'CAL-0010270', 
'CP-5a-838': 'CAL-0010995', 
'CP-5a-841': 'CAL-0010971', 
'CP-5a-850': 'CAL-0010967', 
'CP-8-176': 'CAL-0012008', 
'CP-8-178': 'CAL-0012010'}

data = fs.scraped_data.copy()
data.loc[:, "mapped_name"] = data.molecule_name.map(mol_map)
data.loc[:, "batch"] = 1
data = data.loc[~(data.mapped_name.isna()), :]
data
# data.to_csv('/Users/delafield/Library/CloudStorage/GoogleDrive-delafield@calicolabs.com/My Drive/Projects/ADME_CRO/20251111_PlasmaStability.csv', index=False)

,molecule_name,matrix,time,is_area,cmpd_area,outlier,is_log10_area,cmpd_log10_area,area_ratio,percent_remaining,ln_percent_remaining,rate_constant,linear_fit,half_life,rational_half_life,mapped_name,batch
4,CP-5a-1174,FasSGF + Pepsin,0.0,145286.88,62542.71,False,5.162226,4.796177,0.430477,1.000000,0.000000,-0.0002,0.2411,3465.0,240.0,CAL-0011526,1
5,CP-5a-1174,FasSGF + Pepsin,30.0,152787.38,65283.76,False,5.184087,4.814805,0.427285,0.992584,-0.007443,-0.0002,0.2411,3465.0,240.0,CAL-0011526,1
6,CP-5a-1174,FasSGF + Pepsin,60.0,152215.01,53306.25,False,5.182457,4.726778,0.350204,0.813524,-0.206380,-0.0002,0.2411,3465.0,240.0,CAL-0011526,1
7,CP-5a-1174,FasSGF + Pepsin,240.0,143542.95,57078.91,False,5.156982,4.756476,0.397643,0.923727,-0.079339,-0.0002,0.2411,3465.0,240.0,CAL-0011526,1
8,CP-5a-1174,FasSIF + Pancreatin,0.0,151142.85,39642.33,False,5.179388,4.598159,0.262284,1.000000,0.000000,-0.1665,1.0000,4.2,4.2,CAL-0011526,1
9,CP-5a-1174,FasSIF + Pancreatin,30.0,142354.02,253.01,False,5.153370,2.403138,0.001777,0.006776,-4.994315,-0.1665,1.0000,4.2,4.2,CAL-0011526,1
10,CP-5a-1174,FasSIF + Pancreatin,60.0,161370.58,269.01,True,5.207824,2.429768,0.001667,0.006356,-5.058382,-0.1665,1.0000,4.2,4.2,CAL-0011526,1
11,CP-5a-1174,FasSIF + Pancreatin,240.0,122251.92,98.38,True,5.087256,1.992907,0.000805,0.003068,-5.786674,-0.1665,1.0000,4.2,4.2,CAL-0011526,1
12,CP-5a-1193,FasSGF + Pepsin,0.0,151812.81,61956.94,False,5.181308,4.792090,0.408114,1.000000,0.000000,-0.0001,0.1932,6930.0,240.0,CAL-0011553,1
13,CP-5a-1193,FasSGF + Pepsin,30.0,142246.29,68388.82,False,5.153041,4.834985,0.480778,1.178047,0.163858,-0.0001,0.1932,6930.0,240.0,CAL-0011553,1


In [76]:
import matplotlib.pyplot as plt

In [87]:
d = fs.scraped_data.set_index(["molecule_name", "time"]).sort_index()
d = d.reset_index("time")
# display(d)

l = []

for idx in d.index.unique():
    data = d.loc[idx, :]
    x, y = data.time.to_numpy(), data.ln_percent_remaining.to_numpy()
    print(x)
    print(y)
    # display(data)
    m, b, r, _, _ = linregress(x, y)
    # plt.scatter(data.time, data.ln_percent_remaining)
    # plt.plot(np.linspace(0, 60, 100), np.linspace(0, 60, 100)*m + b)
    # l.append(idx)
    print(linregress(data.time.to_numpy(), data.ln_percent_remaining.to_numpy()))
    # break
    d.loc[idx, "regression_slope"] = m 
    d.loc[idx, "half_life"] = -0.693/m 

# plt.legend(l)
d

[ 0. 15. 30. 60.]
[ 0.         -0.03575245 -0.24760979 -0.65303794]
LinregressResult(slope=-0.011462274053825423, intercept=0.06678464796459854, rvalue=-0.9790752494175289, pvalue=0.020924750582471112, stderr=0.0016846163711640959, intercept_stderr=0.057899115283659694)
[ 0. 15. 30. 60.]
[ 0.         -0.03715852 -0.26693563 -0.47745461]
LinregressResult(slope=-0.008481050497909877, intercept=0.027240384910891507, rvalue=-0.9785209750215745, pvalue=0.021479024978425465, stderr=0.0012634036369990837, intercept_stderr=0.04342232099873109)
[ 0. 15. 30. 60.]
[ 0.         -2.70934344 -4.49691586 -4.33998354]
LinregressResult(slope=-0.0674833093654895, intercept=-1.115123840305996, rvalue=-0.8282570791014612, pvalue=0.1717429208985388, stderr=0.03228302869207879, intercept_stderr=1.109545669829117)
[ 0. 15. 30. 60.]
[ 0.         -0.04706965 -0.04014568 -0.24903714]
LinregressResult(slope=-0.004076706695791768, intercept=0.022950433924405655, rvalue=-0.933092423276033, pvalue=0.066907576723966

,time,is_area,cmpd_area,is_log10_area,cmpd_log10_area,area_ratio,percent_remaining,ln_percent_remaining,regression_slope,half_life
molecule_name,,,,,,,,,,
CP-5a-1174,0.0,148848.17,25364.59,5.172743,4.404228,0.170406,1.000000,0.000000,-0.011462,60.459207
CP-5a-1174,15.0,140952.93,23175.62,5.149074,4.365031,0.164421,0.964879,-0.035752,-0.011462,60.459207
CP-5a-1174,30.0,143289.45,19061.76,5.156214,4.280163,0.133030,0.780665,-0.247610,-0.011462,60.459207
CP-5a-1174,60.0,151541.25,13440.16,5.180531,4.128404,0.088690,0.520462,-0.653038,-0.011462,60.459207
CP-5a-1193,0.0,146539.91,59297.29,5.165956,4.773035,0.404649,1.000000,0.000000,-0.008481,81.711576
CP-5a-1193,15.0,142591.21,55594.77,5.154093,4.745034,0.389889,0.963523,-0.037159,-0.008481,81.711576
CP-5a-1193,30.0,134054.96,41536.81,5.127283,4.618433,0.309849,0.765722,-0.266936,-0.008481,81.711576
CP-5a-1193,60.0,143945.57,36134.44,5.158198,4.557921,0.251028,0.620360,-0.477455,-0.008481,81.711576
CP-5a-1262,0.0,136765.55,6576.38,5.135977,3.817987,0.048085,1.000000,0.000000,-0.067483,10.269206
